# 03 - Rolling (expanding-window) evaluation

Main experiment of the paper: four consecutive test blocks of 155 trading days (620 days in
total). For each block every model is re-estimated on all earlier data.

Models: training-mean constant, persistence, HAR+IV (linear, log), GARCH(1,1) and a 5-seed
TimeMixer ensemble.

**Needs:** `data/eurusd_model_ready.csv`.
**Writes:** `results/rolling_predictions.csv`, `results/rolling_table.csv`.

In [ ]:
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import statsmodels.api as sm
import torch
import torch.nn as nn
from arch import arch_model
from scipy import stats

# The notebooks live in notebooks/, but all paths below are relative to the repository root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# TimeMixer reference implementation (git submodule in TimeMixer/)
sys.path.insert(0, "TimeMixer")
from models.TimeMixer import Model  # noqa: E402

# TimeMixer configuration A: d_model 8, d_ff 16
cfg = SimpleNamespace(
    task_name="long_term_forecast",
    seq_len=64,
    label_len=0,
    pred_len=1,
    enc_in=7,
    c_out=7,
    d_model=8,
    d_ff=16,
    e_layers=2,
    dropout=0.2,
    down_sampling_layers=3,
    down_sampling_window=2,
    down_sampling_method="avg",
    channel_independence=0,
    use_norm=0,
    decomp_method="moving_avg",
    moving_avg=5,
    top_k=5,
    embed="timeF",
    freq="d",
    use_future_temporal_feature=0,
    num_class=0,
)


class Net(nn.Module):
    """TimeMixer core followed by a linear read-out over its 7 output channels."""

    def __init__(self, core):
        super().__init__()
        self.core = core
        self.readout = nn.Linear(7, 1)

    def forward(self, x):
        z = self.core(x, None, None, None)  # (batch, 1, 7)
        return self.readout(z[:, 0, :]).squeeze(-1)  # (batch,)

## 1. Run the four folds

Fold layout (row indices into `eurusd_model_ready.csv`), each separated by a 22-day gap:
`[ train ] gap [ validation: 100 rows ] gap [ test: 155 rows ]`.
The validation rows are used only for early stopping of the neural network.

In [1]:
mdf = pd.read_csv("data/eurusd_model_ready.csv", parse_dates=["Date"])
feat = ["realized_vol_22d", "park_d", "park_w", "park_m", "IMPL_VOL_1M", "VOL_10D", "VOL_90D"]
har_iv = ["park_d", "park_w", "park_m", "IMPL_VOL_1M"]
logF = np.log(mdf[feat]).values
y_all = mdf["target_22d"].values
ly = np.log(y_all)
n, N, GAP, VAL, BLOCK = len(mdf), 64, 22, 100, 155
starts = [620, 775, 930, 1085]
r_pct = mdf.set_index("Date")["log_return"] * 100


def run_fold(a, b, seeds=5):
    tr_end = a - GAP - VAL - GAP
    tr = np.arange(0, tr_end)
    va = np.arange(tr_end + GAP, a - GAP)
    te = np.arange(a, b)

    mu, sd = logF[tr].mean(0), logF[tr].std(0, ddof=1)
    Xs = (logF - mu) / sd
    ymu, ysd = ly[tr].mean(), ly[tr].std(ddof=1)
    ys = (ly - ymu) / ysd

    def win(idx):
        return torch.tensor(np.stack([Xs[i - N + 1 : i + 1] for i in idx]), dtype=torch.float32)

    tr_w = tr[tr >= N - 1]
    Xt, yt_ = win(tr_w), torch.tensor(ys[tr_w], dtype=torch.float32)
    Xv, yv_ = win(va), torch.tensor(ys[va], dtype=torch.float32)
    Xe = win(te)

    out = pd.DataFrame({"Date": mdf["Date"].values[te], "actual": y_all[te], "fold": a})
    out["Train-mean"] = y_all[tr].mean()
    out["Persistence"] = mdf["realized_vol_22d"].values[te]

    m_lin = sm.OLS(y_all[tr], sm.add_constant(mdf[har_iv].iloc[tr])).fit()
    out["HAR+IV (lin)"] = m_lin.predict(
        sm.add_constant(mdf[har_iv].iloc[te], has_constant="add")
    ).values

    lc = np.log(mdf[har_iv])
    m_log = sm.OLS(ly[tr], sm.add_constant(lc.iloc[tr])).fit()
    out["HAR+IV (log)"] = np.exp(
        m_log.predict(sm.add_constant(lc.iloc[te], has_constant="add"))
    ).values

    am = arch_model(r_pct, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
    res_g = am.fit(last_obs=mdf["Date"].iloc[a - GAP - 1], disp="off")
    fc = res_g.forecast(horizon=22, start=mdf["Date"].iloc[a], reindex=False)
    g = np.sqrt(fc.variance.mean(axis=1) * 252)
    out["GARCH"] = g.reindex(pd.DatetimeIndex(mdf["Date"].iloc[te])).values

    preds = []
    for s in range(seeds):
        torch.manual_seed(s)
        np.random.seed(s)
        net = Net(Model(cfg))
        opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
        lf = nn.MSELoss()
        best, best_state, bad = float("inf"), None, 0
        for ep in range(100):
            net.train()
            perm = torch.randperm(len(Xt))
            for i in range(0, len(perm), 32):
                ix = perm[i : i + 32]
                opt.zero_grad()
                lf(net(Xt[ix]), yt_[ix]).backward()
                opt.step()
            net.eval()
            with torch.no_grad():
                v = lf(net(Xv), yv_).item()
            if v < best - 1e-5:
                best, bad = v, 0
                best_state = {k: x.clone() for k, x in net.state_dict().items()}
            else:
                bad += 1
                if bad >= 10:
                    break
        net.load_state_dict(best_state)
        net.eval()
        with torch.no_grad():
            preds.append(np.exp(net(Xe).numpy() * ysd + ymu))
    out["TimeMixer ens5"] = np.mean(preds, axis=0)
    for s, p in enumerate(preds):
        out[f"tm_seed{s}"] = p
    return out


folds = []
for a in starts:
    b = min(a + BLOCK, n)
    print("fold:", mdf["Date"].iloc[a].date(), "->", mdf["Date"].iloc[b - 1].date(), flush=True)
    folds.append(run_fold(a, b))

roll = pd.concat(folds, ignore_index=True)
roll.to_csv("results/rolling_predictions.csv", index=False)
print(roll.shape, "| NaNs:", int(roll.isnull().sum().sum()))

fold: 2024-04-09 -> 2024-11-11
fold: 2024-11-12 -> 2025-06-16
fold: 2025-06-17 -> 2026-01-19
fold: 2026-01-20 -> 2026-08-24
(620, 14) | NaNs: 0


## 2. Pooled and per-fold accuracy

In [2]:
meta = ["Date", "actual", "fold"]
models = [c for c in roll.columns if c not in meta and not c.startswith("tm_seed")]
y = roll["actual"].values


def qlike(y, p):
    r = (y**2) / (p**2)
    return np.mean(r - np.log(r) - 1)


tab = pd.DataFrame(
    {
        m: {
            "RMSE": np.sqrt(np.mean((y - roll[m]) ** 2)),
            "MAE": np.mean(np.abs(y - roll[m])),
            "bias": np.mean(roll[m] - y),
            "QLIKE": qlike(y, roll[m].values),
            "corr": np.corrcoef(roll[m], y)[0, 1],
        }
        for m in models
    }
).T.round(3)
print("POOLED (all 4 test blocks)\n", tab, "\n")

byfold = pd.DataFrame(
    {
        f: {m: np.sqrt(np.mean((g["actual"] - g[m]) ** 2)) for m in models}
        for f, g in roll.groupby("fold")
    }
).T.round(3)
print("RMSE BY FOLD (index = fold start row)\n", byfold, "\n")

single = [np.sqrt(np.mean((y - roll[f"tm_seed{s}"]) ** 2)) for s in range(5)]
print(
    "TimeMixer single-seed pooled RMSE: mean",
    round(np.mean(single), 3),
    "std",
    round(np.std(single), 3),
    "\n",
)

tab.to_csv("results/rolling_table.csv")

POOLED (all 4 test blocks)
                  RMSE    MAE   bias  QLIKE   corr
Train-mean      2.728  2.354  1.466  0.275  0.006
Persistence     2.332  1.860  0.063  0.329  0.467
HAR+IV (lin)    1.855  1.487  0.453  0.153  0.624
HAR+IV (log)    1.828  1.440  0.305  0.152  0.620
GARCH           2.145  1.845  0.713  0.196  0.468
TimeMixer ens5  1.967  1.532 -0.047  0.193  0.505 

RMSE BY FOLD (index = fold start row)
       Train-mean  Persistence  HAR+IV (lin)  HAR+IV (log)  GARCH  \
620        3.180        2.207         1.641         1.597  1.883   
775        2.266        2.620         2.206         2.217  2.406   
930        2.674        2.613         1.982         1.950  2.420   
1085       2.714        1.787         1.509         1.447  1.794   

      TimeMixer ens5  
620            1.721  
775            2.372  
930            2.129  
1085           1.532   

TimeMixer single-seed pooled RMSE: mean 2.032 std 0.057 



## 3. Diebold-Mariano tests with the Harvey-Leybourne-Newbold correction

Squared-error loss differential, 21-lag Bartlett HAC variance, small-sample correction for
horizon h = 22 and Student-t reference distribution. A positive statistic means the second
model is more accurate.

In [3]:
roll = pd.read_csv("results/rolling_predictions.csv", parse_dates=["Date"])
y = roll["actual"].values


def dm_hln(y, p1, p2, h=22, lags=21):
    d = (y - p1) ** 2 - (y - p2) ** 2  # positive => model 2 better
    T = len(d)
    e = d - d.mean()
    var = (e @ e) / T
    for k in range(1, lags + 1):
        var += 2 * (1 - k / (lags + 1)) * (e[k:] @ e[:-k]) / T
    stat = d.mean() / np.sqrt(var / T)
    stat *= np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)  # HLN correction
    return stat, 2 * (1 - stats.t.cdf(abs(stat), T - 1))


pairs = [
    ("GARCH", "HAR+IV (log)"),
    ("Train-mean", "HAR+IV (log)"),
    ("Train-mean", "TimeMixer ens5"),
    ("GARCH", "TimeMixer ens5"),
    ("HAR+IV (log)", "TimeMixer ens5"),
    ("HAR+IV (lin)", "TimeMixer ens5"),
]
for a, b in pairs:
    s, p = dm_hln(y, roll[a].values, roll[b].values)
    print(f"DM-HLN  {a:14s} vs {b:14s}: stat {s:6.2f}, p {p:.3f}")

DM-HLN  GARCH          vs HAR+IV (log)  : stat   2.87, p 0.004
DM-HLN  Train-mean     vs HAR+IV (log)  : stat   4.23, p 0.000
DM-HLN  Train-mean     vs TimeMixer ens5: stat   2.89, p 0.004
DM-HLN  GARCH          vs TimeMixer ens5: stat   1.60, p 0.111
DM-HLN  HAR+IV (log)   vs TimeMixer ens5: stat  -1.11, p 0.268
DM-HLN  HAR+IV (lin)   vs TimeMixer ens5: stat  -0.82, p 0.413
